# 임베딩 입력 기준 비교 실험

목적: 동일한 청크·동일한 임베딩 모델·동일한 Vector DB에서 **임베딩 입력 텍스트만** 바꾸어 검색 성능을 비교한다.

- B (`era_primary_type`): `title + section + era + primary_type + content`
- C (`with_aliases`): B에 `aliases` 추가

`chunk_id`, URL, 라이선스, 수정일은 의미 검색용 문장이 아니므로 임베딩하지 않고 Pinecone metadata로만 저장한다.

## 실험 원칙

1. 청킹 설정, 모델, `top_k=3`, 질문 세트는 고정한다.
2. 질문마다 정답 원문 `document_id`를 미리 기록한다.
3. `Recall@3`(정답 문서가 상위 3개에 포함된 비율), `MRR`(정답 순위), 시간·토큰 사용량을 비교한다.
4. 성능 차이가 작으면 더 짧고 단순한 입력안을 선택한다.

처음에는 `RUN_API_EXPERIMENT=False` 상태로 미리보기만 한다. API 비용이 발생하는 적재는 팀 합의 후에만 `True`로 바꾼다.

In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
import time
from dotenv import load_dotenv
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

CHUNKS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'aks_chunks.jsonl'
RESULTS_DIR = PROJECT_ROOT / 'outputs'
RESULTS_DIR.mkdir(exist_ok=True)

load_dotenv(PROJECT_ROOT / '.env', override=False)

# 비용이 발생하는 API 적재 방지 장치
RUN_API_EXPERIMENT = False
# 정답 문서와 함께 넣을 무작위 방해 청크 수. 전체 179,028청크를 재임베딩하지 않는 비용 절약형 파일럿이다.
RANDOM_DISTRACTOR_CHUNKS = 2_000
PILOT_TARGET_DOCUMENT_IDS = {
    'aks:E0000003', 'aks:E0008547', 'aks:E0008548',
    'aks:E0009225', 'aks:E0009227', 'aks:E0009229',
}
TOP_K = 3
# 새 서버리스 인덱스를 만들지 않고, 기존 인덱스의 평가용 namespace를 사용한다.
EVALUATION_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME', 'aks-rag-v1')
EVALUATION_NAMESPACE_PREFIX = 'embedding-input-alias-pilot-v1'
EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_DIMENSION = 1536

print({'chunks_path': str(CHUNKS_PATH), 'run_api_experiment': RUN_API_EXPERIMENT, 'random_distractor_chunks': RANDOM_DISTRACTOR_CHUNKS})

In [ ]:
def build_pilot_corpus(path: Path, target_document_ids: set[str], random_chunk_count: int, seed: int = 33) -> list[dict]:
    """Keep every target-document chunk plus a deterministic reservoir sample of distractors."""
    targets, reservoir = [], []
    randomizer = random.Random(seed)
    distractors_seen = 0
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            chunk = json.loads(line)
            if chunk['document_id'] in target_document_ids:
                targets.append(chunk)
                continue
            distractors_seen += 1
            if len(reservoir) < random_chunk_count:
                reservoir.append(chunk)
            else:
                replacement = randomizer.randrange(distractors_seen)
                if replacement < random_chunk_count:
                    reservoir[replacement] = chunk
    return targets + reservoir

chunks = build_pilot_corpus(CHUNKS_PATH, PILOT_TARGET_DOCUMENT_IDS, RANDOM_DISTRACTOR_CHUNKS)
print({'pilot_chunks': len(chunks), 'target_document_ids': len(PILOT_TARGET_DOCUMENT_IDS), 'random_distractors': RANDOM_DISTRACTOR_CHUNKS})

SECTION_LABEL = {'definition': '정의', 'body': '상세 본문'}

def embedding_input(chunk: dict, variant: str) -> str:
    metadata = chunk['metadata']
    lines = [
        f"제목: {chunk['title']}",
        f"구역: {SECTION_LABEL.get(chunk.get('section'), chunk.get('section') or '미상')}",
    ]
    if variant in {'era_primary_type', 'with_aliases'}:
        if metadata.get('era'):
            lines.append(f"시대: {metadata['era']}")
        if metadata.get('primary_type'):
            lines.append(f"유형: {metadata['primary_type']}")
    if variant == 'with_aliases' and metadata.get('aliases'):
        lines.append(f"이칭: {', '.join(metadata['aliases'])}")
    lines.append(f"본문: {chunk['content']}")
    return '\n'.join(lines)

VARIANTS = ['era_primary_type', 'with_aliases']
for variant in VARIANTS:
    print(f'\n[{variant}]\n{embedding_input(chunks[0], variant)[:500]}')

In [ ]:
# 각 질문의 정답 document_id는 사람이 원문을 확인해 기록한다.
# 최종 평가 전 정의형·세부 사실형·설명형·동명이인 구분형을 포함해 20~30개로 늘린다.
EVALUATION_SET = [
    {'id': 'Q01', 'question': 'ㄱ당은 어떤 단체야?', 'expected_document_id': 'aks:E0000003', 'type': '정의형'},
    {'id': 'Q02', 'question': '길쌈노래에 대해 설명해줘', 'expected_document_id': 'aks:E0008547', 'type': '설명형'},
    {'id': 'Q03', 'question': '길씨세효록은 어떤 문헌이야?', 'expected_document_id': 'aks:E0008548', 'type': '정의형'},
    {'id': 'Q04', 'question': '김병학은 어떤 독립운동 활동을 했어?', 'expected_document_id': 'aks:E0009225', 'type': '세부 사실형'},
    {'id': 'Q05', 'question': '김병호 고가는 어디에 있는 건물이야?', 'expected_document_id': 'aks:E0009227', 'type': '정의형'},
    {'id': 'Q06', 'question': '김보는 어느 시대의 문신이야?', 'expected_document_id': 'aks:E0009229', 'type': '시대·유형형'},
]

available_document_ids = {chunk['document_id'] for chunk in chunks}
available_questions = [q for q in EVALUATION_SET if q['expected_document_id'] in available_document_ids]
missing_questions = [q['id'] for q in EVALUATION_SET if q not in available_questions]
print({'questions_available_in_current_corpus': len(available_questions), 'missing_question_ids': missing_questions})
available_questions

## API 실험 실행 셀

아래 셀은 기존 Pinecone 인덱스 안에 variant별 평가용 namespace를 만든다. `RUN_API_EXPERIMENT=True`일 때만 실행한다. 서비스 기본 namespace와는 분리되므로 기존 벡터를 덮어쓰지 않는다.

파일럿은 평가 질문의 정답 문서 전체 청크와 무작위 방해 청크 2,000개로 구성한다. 두 variant는 같은 파일럿 코퍼스·같은 모델·같은 `top_k`에서 비교하므로 aliases 추가 효과만 비교할 수 있다.

In [ ]:
# %pip install pinecone openai python-dotenv

In [ ]:
def flat_metadata(chunk: dict) -> dict:
    return {
        'document_id': chunk['document_id'],
        'title': chunk['title'],
        'content': chunk['content'],
        'source_url': chunk['source_url'] or '',
        'section': chunk['section'] or '',
        **{key: value for key, value in chunk['metadata'].items() if value is not None},
    }

def run_variant(variant: str, corpus: list[dict], batch_size: int = 100) -> dict:
    from dotenv import load_dotenv
    from openai import OpenAI
    from pinecone import Pinecone

    load_dotenv(PROJECT_ROOT / '.env', override=False)
    openai_client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    pinecone = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
    if EVALUATION_INDEX_NAME not in pinecone.list_indexes().names():
        raise RuntimeError(f'기존 Pinecone 인덱스를 찾지 못했습니다: {EVALUATION_INDEX_NAME}')
    index = pinecone.Index(EVALUATION_INDEX_NAME)
    namespace = f'{EVALUATION_NAMESPACE_PREFIX}-{variant}'

    started = time.perf_counter()
    input_tokens = 0
    for start in range(0, len(corpus), batch_size):
        batch = corpus[start:start + batch_size]
        response = openai_client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=[embedding_input(chunk, variant) for chunk in batch],
        )
        input_tokens += getattr(response.usage, 'total_tokens', 0)
        index.upsert(
            namespace=namespace,
            vectors=[
                {'id': chunk['chunk_id'], 'values': item.embedding, 'metadata': flat_metadata(chunk)}
                for chunk, item in zip(batch, response.data, strict=True)
            ],
        )
    return {
        'variant': variant,
        'namespace': namespace,
        'chunks_indexed': len(corpus),
        'embedding_input_tokens': input_tokens,
        'indexing_seconds': round(time.perf_counter() - started, 2),
    }

if RUN_API_EXPERIMENT:
    indexing_runs = [run_variant(variant, chunks) for variant in VARIANTS]
    indexing_runs
else:
    print('API 호출 없음: RUN_API_EXPERIMENT=True로 바꾼 뒤 실행하세요.')

In [ ]:
def evaluate_variant(variant: str, questions: list[dict]) -> dict:
    from dotenv import load_dotenv
    from openai import OpenAI
    from pinecone import Pinecone

    load_dotenv(PROJECT_ROOT / '.env', override=False)
    openai_client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    index = Pinecone(api_key=os.environ['PINECONE_API_KEY']).Index(EVALUATION_INDEX_NAME)
    rows = []
    for item in questions:
        started = time.perf_counter()
        query_vector = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=[item['question']]).data[0].embedding
        namespace = f'{EVALUATION_NAMESPACE_PREFIX}-{variant}'
        response = index.query(vector=query_vector, namespace=namespace, top_k=TOP_K, include_metadata=True)
        found_rank = None
        for rank, match in enumerate(response.matches, start=1):
            if match.metadata.get('document_id') == item['expected_document_id']:
                found_rank = rank
                break
        rows.append({
            **item,
            'variant': variant,
            'found_rank': found_rank,
            'hit_at_3': found_rank is not None,
            'reciprocal_rank': 0 if found_rank is None else 1 / found_rank,
            'query_seconds': round(time.perf_counter() - started, 3),
        })
    return {
        'variant': variant,
        'recall_at_3': sum(row['hit_at_3'] for row in rows) / len(rows),
        'mrr': sum(row['reciprocal_rank'] for row in rows) / len(rows),
        'mean_query_seconds': sum(row['query_seconds'] for row in rows) / len(rows),
        'rows': rows,
    }

if RUN_API_EXPERIMENT:
    evaluation_runs = [evaluate_variant(variant, available_questions) for variant in VARIANTS]
    summary = [{key: value for key, value in run.items() if key != 'rows'} for run in evaluation_runs]
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    result_path = RESULTS_DIR / f"embedding_input_eval_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
    result_path.write_text(json.dumps({'indexing': indexing_runs, 'evaluation': evaluation_runs}, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'written: {result_path}')
else:
    print('API 호출 없음: 위 적재 셀 완료 후 이 셀을 실행하세요.')

## 팀 의사결정 기록

- `Recall@3`와 `MRR`이 높은 입력안을 우선 선택한다.
- 지표 차이가 작으면 metadata가 적은 더 단순한 입력안을 선택한다.
- 이번 파일럿은 `era_primary_type`과 `with_aliases`를 비교한다.
- `with_aliases`가 Recall@3 또는 MRR을 개선하지 못하면 aliases를 임베딩 입력에서 제외한다.
- 최종 입력안과 모델을 확정한 뒤에만 전체 179,028개 청크를 서비스용 Pinecone 인덱스에 적재한다.

---

era + primary_type:
- Recall@3 = 1.0
- MRR = 0.917

era + primary_type + aliases:
- Recall@3 = 1.0
- MRR = 1.0

### 임베딩 후보: title + section + era + primary_type + aliases + content